In [2]:

import pandas as pd
pd.set_option('display.max_columns', None)


import numpy as np


import seaborn as sns
import matplotlib.pyplot as plt


import warnings
warnings.filterwarnings('ignore')

In [3]:
# Load the cleaned and merged dataset for statistical analysis and visualization
df_clean = pd.read_csv("../data/Customer_Flight_Loyalty_Clean.csv")

In [4]:
df_clean.sample(5)

,Loyalty Number,Year,Month,Flights Booked,Flights with Companions,Total Flights,Distance,Points Accumulated,Country,Province,City,Postal Code,Gender,Education,Salary,Marital Status,Loyalty Card,CLV,Enrollment Type,Enrollment Year,Enrollment Month,Customer Active
76978,216139,2017,10,8,4,12,780,78.0,Canada,Manitoba,Winnipeg,R2C 0M5,Male,Bachelor,98586.000000,Divorced,Aurora,10883.79,Standard,2014,3,True
377608,846055,2017,12,0,0,0,0,0.0,Canada,British Columbia,Vancouver,V5R 1W3,Male,Doctor,274242.000000,Married,Star,10418.31,Standard,2014,8,True
310317,135977,2017,4,8,4,12,3804,380.0,Canada,Manitoba,Winnipeg,R2C 0M5,Male,Bachelor,68601.000000,Married,Star,4514.25,Standard,2016,9,True
125183,109660,2017,10,0,0,0,0,0.0,Canada,Alberta,Edmonton,T3G 6Y6,Male,Bachelor,58026.000000,Married,Nova,3543.23,Standard,2016,2,True
208186,182531,2017,10,0,0,0,0,0.0,Canada,British Columbia,Vancouver,V6E 3Z3,Female,College,78353.298918,Single,Nova,9672.70,Standard,2015,12,True


## 01. Numerical Variable Analysis

### Descriptive Statistics: 

- Calculate summary statistics (mean, median, mode, standard deviation, etc.) for the relevant numerical variables.

In [12]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 403760 entries, 0 to 403759
Data columns (total 22 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   Loyalty Number           403760 non-null  int64  
 1   Year                     403760 non-null  int64  
 2   Month                    403760 non-null  int64  
 3   Flights Booked           403760 non-null  int64  
 4   Flights with Companions  403760 non-null  int64  
 5   Total Flights            403760 non-null  int64  
 6   Distance                 403760 non-null  int64  
 7   Points Accumulated       403760 non-null  float64
 8   Country                  403760 non-null  str    
 9   Province                 403760 non-null  str    
 10  City                     403760 non-null  str    
 11  Postal Code              403760 non-null  str    
 12  Gender                   403760 non-null  str    
 13  Education                403760 non-null  str    
 14  Salary         

In [28]:
# Select the relevant numerical variables for statistical analysis
# Year/Month-type column are excluded because they act as grouping variables rather than numerical distributions
# Loyalty Number is excluded as it is a customer identifier, not a metric.

numerical_cols_flights = ['Flights Booked', 'Flights with Companions', 'Total Flights', 'Distance', 'Points Accumulated']

round(df_clean[numerical_cols_flights].describe(), 2).T.reset_index()

,index,count,mean,std,min,25%,50%,75%,max
0,Flights Booked,403760.0,4.13,5.23,0.0,0.0,1.0,8.0,21.0
1,Flights with Companions,403760.0,1.04,2.08,0.0,0.0,0.0,1.0,11.0
2,Total Flights,403760.0,5.17,6.53,0.0,0.0,1.0,10.0,32.0
3,Distance,403760.0,1214.46,1434.10,0.0,0.0,525.0,2342.0,6293.0
4,Points Accumulated,403760.0,124.26,146.70,0.0,0.0,53.0,240.0,676.5


In [29]:
# Calculate the mode for the relevant numerical variables
df_clean[numerical_cols_flights].mode().iloc[0]

Flights Booked             0.0
Flights with Companions    0.0
Total Flights              0.0
Distance                   0.0
Points Accumulated         0.0
Name: 0, dtype: float64

***Initial Insights & Real-World Meaning:***

* **Flights Booked & Flights with Companions:** 
    - `Flights Booked` has a mean of 4.13 but a high standard deviation of 5.23, with half of the records (up to the 50% percentile) showing 1 or fewer flights. This gap between the median and the maximum value of 21 suggests the presence of high-value outliers, that indicates a specific profile: **frequent business travelers** who fly constantly.
    - `Flights with Companions`, the maximum is 11, while the mean is only 1.04. The value 0 is highly recurrent up to the 50% percentile, showing that **the vast majority of customers travel alone**.

* **Total Flights:** As the combination of both metrics, it reflects a similar behavior. The first two quartiles hold a value of 1.00, and the maximum spikes to 32, which strongly points to outliers in flight frequency. This confirms that **the airline's baseline activity relies heavily on a small group of highly active users** who push the standard deviation up to 6.53.

* **Distance & Points Accumulated:** Both variables behave almost identically because points depend directly on distance. Both present a very high variability (their standard deviations are larger than their means). This explains why the first 25% of the data is 0, proving that **in any given month, a significant portion of program members do not fly at all**, while the maximum values stretch far out to 6,293 units in distance and 676.50 in points, indicating clear outliers from heavy travelers that make **long-haul international flights**


* **Mode Analysis:** Looking at the most frequent values (mode), we can confirm that for any given month, the most common scenario for a customer is to have `0` flights booked, `0` flights with companions, `0` total flights, `0` distance flown, and `0` points accumulated. This aligns with our previous observation that a large portion of the loyalty program members remain **inactive during many months of the year**.

**Methodological Note:** 

Since the dataset contains monthly records, client-specific variables like `Salary` and `CLV` are repeated across months. To ensure accurate demographic analysis without inflation, a sub-analysis of unique customers was conducted, confirming that the overall income and valuation averages remain stable.

In [30]:
# Group by customer to analyze unique profile variables (Salary and CLV) without monthly repetition
df_unique_customers = df_clean.drop_duplicates(subset=['Loyalty Number'])
numerical_cols_customers = ['Salary', 'CLV']
round(df_unique_customers[numerical_cols_customers].describe(),2).T.reset_index()

,index,count,mean,std,min,25%,50%,75%,max
0,Salary,16737.0,79362.93,30032.44,9081.00,63899.00,78879.00,82940.00,407228.00
1,CLV,16737.0,7988.90,6860.98,1898.01,3980.84,5780.18,8940.58,83325.38


In [31]:
# Calculate the mode for the relevant numerical variables

df_unique_customers[numerical_cols_customers].mode().iloc[0]

Salary    101933.00
CLV         8564.77
Name: 0, dtype: float64

***Initial Insights & Real-World Meaning (for Unique Customer):***

* **Salary:** The average salary is around \$79,362.93 and is very stable for 75% of the dataset (under \$82,940.00). However, the maximum of \$407,228.00 reveals **a small group of high-income, wealthy clients**. This tells us the loyalty program successfully attracts both middle-class passengers and high-net-worth individuals.

* **Customer Lifetime Value (CLV):** While most customers bring a standard value to the airline (75% are below \$8,940.58), the maximum customer value reaches \$83,325.38. This proves the **existence of a VIP tier of premium customers** who are exceptionally profitable for the company and worth targeting with special marketing strategies.


* **Mode Analysis (Unique Customers):** Interestingly, the most frequent salary (mode) is \$101,933.00, which sits well above the 75th percentile of the dataset. Similarly, the most common Customer Lifetime Value (CLV) is \$8,564.77, positioning itself very close to the top 25% of customers. This indicates that while there is a wide distribution of profiles, the loyalty program features a highly concentrated and repetitive segment of high-income, high-value premium users.

### Outlier Identification:
- Detect and analyze outlier values within the numerical variables.

In [32]:
# Identify and count the number of outliers in the specified numerical columns using the IQR method
def count_outliers(df, column_list):

    dictionary = {}
    for col in column_list:
        
        # Calculate quartiles and Interquartile Range (IQR)
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        # Define outlier thresholds (1.5 * IQR rule)
        lower = Q1  - IQR*1.5
        upper = Q3 + IQR*1.5

        # Extract and count rows outside the thresholds
        outliers = df[(df[col]<lower) | (df[col]>upper)]
        dictionary[col] = len(outliers)
    return dictionary

In [ ]:
# Execute the outlier detection function on the flight-related numerical columns
flight_outliers = count_outliers(df_clean, numerical_cols_flights)

# Print the results in a clean, formatted table
print("Outlier Count per Variable:")
print("-" * 30)
for column, count in flight_outliers.items():
    print(f"{column:<25} : {count:>6} outliers")

Outlier Count per Variable:
------------------------------
Flights Booked            :    528 outliers
Flights with Companions   :  71560 outliers
Total Flights             :   1984 outliers
Distance                  :    125 outliers
Points Accumulated        :    112 outliers


In [40]:
# Execute the outlier detection function on the customer-related numerical columns
customers_outliers = count_outliers(df_unique_customers, numerical_cols_customers)

# Print the results in a clean, formatted table
print("Outlier Count per Variable:")
print("-" * 30)
for column, count in customers_outliers.items():
    print(f"{column:<25} : {count:>6} outliers")

Outlier Count per Variable:
------------------------------
Salary                    :    866 outliers
CLV                       :   1485 outliers


**Outlier Analysis and Detection:**

* **General Overview:** Outliers are present across all studied numerical variables. The column with the lowest number of extreme values is `Points Accumulated` (112), while the highest is `Flights with Companions` (71,560).

* **Flights with Companions:** This massive count highlights that, as shown by the 75th percentile (1.00), traveling with companions is quite rare. The statistical threshold sets the upper limit at 2.5, meaning that while making 1 or 2 companion flights is considered standard behavior, **any record with 3 or more companions is immediately flagged as an outlier**.

* **Total Flights:** The outlier count drops significantly to 1,984 records. This still indicates that a high frequency of monthly trips is uncommon and stands out from the baseline behavior of regular passengers.

* **Salary vs. Customer Lifetime Value (CLV):** There is a small group of unique customers with high salaries that fall well outside the normal range, confirming the presence of an affluent segment. Interestingly, this outlier count significantly increases for `CLV`. This proves that a high income is **not the only factor that makes a customer valuable to the airline**; engagement, distance flown, and loyalty play a massive role in driving up a customer's commercial worth.

### Correlation Analysis:

- Examine the relationships and correlation between numerical variables.